# Lectura e interpretación de matrices dispersas (TF-IDF) — con varios ejemplos de reseñas y Naive Bayes

**Objetivo:** entender a fondo la estructura interna de una matriz dispersa (formato CSR), leerla directamente (sin "densificar") usando varios ejemplos concretos de reseñas, y conectar esa estructura con cómo **Naive Bayes Multinomial** la aprovecha internamente.

Por indicación tuya, en este notebook usamos **únicamente el modelo Naive Bayes** (sin comparar con Regresión Logística o SVM, como sí se hizo en el notebook 1).

In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
%matplotlib inline

## 1. Preparación de datos

Replicamos brevemente el mismo pipeline del notebook 1 (limpieza, muestreo balanceado, split, TF-IDF) para que este notebook sea autocontenido y comparable con los anteriores.

In [2]:
df = pd.read_csv('data/IMDB Dataset.csv')

# Eliminamos las 418 filas duplicadas confirmadas en el notebook 2 (texto 100% idéntico)
df = df.drop_duplicates().reset_index(drop=True)

N = 5000
df_pos = df[df['sentiment'] == 'positive'].sample(n=N, random_state=42)
df_neg = df[df['sentiment'] == 'negative'].sample(n=N, random_state=42)
df_reviews = pd.concat([df_pos, df_neg]).reset_index(drop=True)

from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df_reviews, test_size=0.33, random_state=42, stratify=df_reviews['sentiment']
)
train_x, train_y = train['review'].reset_index(drop=True), train['sentiment'].reset_index(drop=True)
test_x, test_y = test['review'].reset_index(drop=True), test['sentiment'].reset_index(drop=True)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

vocabulario = tfidf.get_feature_names_out()
print(f"Vocabulario: {len(vocabulario):,} términos")
print(f"Matriz train: {train_x_vector.shape} | Matriz test: {test_x_vector.shape}")
print(f"Tipo de objeto: {type(train_x_vector)}")

Vocabulario: 43,695 términos
Matriz train: (6700, 43695) | Matriz test: (3300, 43695)
Tipo de objeto: <class 'scipy.sparse._csr.csr_matrix'>


## 2. ¿Qué es realmente una matriz CSR por dentro?

`TfidfVectorizer` no entrega una tabla normal: entrega un objeto `scipy.sparse.csr_matrix` (*Compressed Sparse Row*). En vez de guardar los $m \times n$ números (la inmensa mayoría ceros), guarda **solo 3 arreglos**:

- **`data`**: los valores distintos de cero, en orden.
- **`indices`**: la columna (término del vocabulario) a la que pertenece cada valor de `data`.
- **`indptr`**: un arreglo de $m+1$ posiciones que indica **dónde empieza y termina cada fila** dentro de `data`/`indices`.

Para leer la fila $i$: sus valores no-cero son `data[indptr[i] : indptr[i+1]]`, ubicados en las columnas `indices[indptr[i] : indptr[i+1]]`.

Esto reduce la memoria de $O(m \times n)$ a $O(\text{nnz})$ (nnz = número de elementos distintos de cero) — la diferencia es enorme cuando, como veremos, más del 99% de la matriz son ceros.

In [3]:
print(f"nnz (elementos distintos de cero): {train_x_vector.nnz:,}")
print(f"data[:10]    -> {train_x_vector.data[:10]}")
print(f"indices[:10] -> {train_x_vector.indices[:10]}")
print(f"indptr[:5]   -> {train_x_vector.indptr[:5]}")

bytes_sparse = train_x_vector.data.nbytes + train_x_vector.indices.nbytes + train_x_vector.indptr.nbytes
bytes_denso = train_x_vector.shape[0] * train_x_vector.shape[1] * 8  # float64 = 8 bytes

print(f"\nMemoria como matriz dispersa: {bytes_sparse/1e6:.2f} MB")
print(f"Memoria si se guardara como matriz densa: {bytes_denso/1e6:.2f} MB")
print(f"Ahorro: {(1 - bytes_sparse/bytes_denso)*100:.2f}%")

nnz (elementos distintos de cero): 595,778
data[:10]    -> [0.05953243 0.04819392 0.1087527  0.21943164 0.05242967 0.16959368
 0.1124966  0.04046725 0.04098514 0.04174362]
indices[:10] -> [37958 14437 33138  1810 22107 14415 22849 12194 32467 31423]
indptr[:5]   -> [  0 268 334 398 440]

Memoria como matriz dispersa: 7.18 MB
Memoria si se guardara como matriz densa: 2342.05 MB
Ahorro: 99.69%


**Verificación:** reconstruimos manualmente la fila 0 usando `indptr` y la comparamos con la forma normal de acceder a esa fila, para confirmar que entendemos la estructura.

In [4]:
inicio, fin = train_x_vector.indptr[0], train_x_vector.indptr[1]
columnas_fila0 = train_x_vector.indices[inicio:fin]
valores_fila0 = train_x_vector.data[inicio:fin]

print(f"La fila 0 tiene {fin - inicio} valores distintos de cero.")
print("Reconstrucción manual (columna -> valor), primeros 5:")
for col, val in list(zip(columnas_fila0, valores_fila0))[:5]:
    print(f"  columna {col} ('{vocabulario[col]}') -> {val:.4f}")

# Comprobación: debe coincidir exactamente con acceder a la fila con la sintaxis normal
fila0_normal = train_x_vector[0]
assert np.allclose(fila0_normal.data, valores_fila0) and np.array_equal(fila0_normal.indices, columnas_fila0)
print("\n✔ La reconstrucción manual coincide exactamente con train_x_vector[0].")

La fila 0 tiene 268 valores distintos de cero.
Reconstrucción manual (columna -> valor), primeros 5:
  columna 37958 ('swedish') -> 0.0595
  columna 14437 ('filmmaker') -> 0.0482
  columna 33138 ('roy') -> 0.1088
  columna 1810 ('andersson') -> 0.2194
  columna 22107 ('latest') -> 0.0524

✔ La reconstrucción manual coincide exactamente con train_x_vector[0].


## 3. Varios ejemplos de reseñas — leyendo sus vectores dispersos

Elegimos 5 reseñas distintas (la más corta, la más larga, una de longitud mediana, y una positiva y otra negativa al azar) para leer su vector TF-IDF **directamente desde la matriz dispersa**, sin convertir nada a denso.

**Nota importante:** el número de términos no-cero de una reseña es **el vocabulario único** que usa (después de quitar *stopwords*), no el conteo total de palabras — si una palabra se repite 5 veces, sigue ocupando **una sola columna** (con un valor TF-IDF más alto), no 5.

In [5]:
longitudes = train_x.apply(lambda x: len(x.split()))
idx_corta = int(longitudes.values.argmin())
idx_larga = int(longitudes.values.argmax())
idx_mediana = int(np.argsort(longitudes.values)[len(longitudes) // 2])

rng = np.random.default_rng(42)
idx_pos = int(rng.choice(np.where(train_y.values == 'positive')[0]))
idx_neg = int(rng.choice(np.where(train_y.values == 'negative')[0]))

ejemplos = {
    'Reseña más corta': idx_corta,
    'Reseña más larga': idx_larga,
    'Reseña de longitud mediana': idx_mediana,
    'Ejemplo positivo (aleatorio)': idx_pos,
    'Ejemplo negativo (aleatorio)': idx_neg,
}

for nombre, idx in ejemplos.items():
    texto = train_x.iloc[idx]
    sentimiento = train_y.iloc[idx]
    fila = train_x_vector[idx]
    n_palabras = len(texto.split())

    print(f"=== {nombre} (sentiment={sentimiento}, id={idx}) ===")
    print(f"Texto ({n_palabras} palabras totales, primeros 180 caracteres):")
    print(f"  {texto[:180]}...")
    print(f"Términos con TF-IDF != 0: {fila.nnz} de {len(vocabulario):,} posibles ({fila.nnz/len(vocabulario)*100:.3f}% del vocabulario)")

    top = pd.Series(fila.data, index=vocabulario[fila.indices]).sort_values(ascending=False).head(8)
    print("Términos con mayor peso TF-IDF:")
    print(top.to_string())
    print()

=== Reseña más corta (sentiment=negative, id=794) ===
Texto (4 palabras totales, primeros 180 caracteres):
  Primary plot!Primary direction!Poor interpretation....
Términos con TF-IDF != 0: 5 de 43,695 posibles (0.011% del vocabulario)
Términos con mayor peso TF-IDF:
primary           0.835422
interpretation    0.391861
direction         0.257475
poor              0.233711
plot              0.166134

=== Reseña más larga (sentiment=positive, id=4489) ===
Texto (2278 palabras totales, primeros 180 caracteres):
  There's a sign on The Lost Highway that says:<br /><br />*MAJOR SPOILERS AHEAD*<br /><br />(but you already knew that, didn't you?)<br /><br />Since there's a great deal of people ...
Términos con TF-IDF != 0: 549 de 43,695 posibles (1.256% del vocabulario)
Términos con mayor peso TF-IDF:
diane      0.544526
br         0.348992
camilla    0.341725
dream      0.220450
betty      0.190866
adam       0.141876
hitman     0.131061
lynch      0.112355

=== Reseña de longitud mediana (

## 4. Similitud entre reseñas usando directamente los vectores dispersos

Como cada vector TF-IDF está normalizado a **norma L2 = 1**, el **producto punto entre dos vectores es exactamente su similitud coseno** — no hace falta ninguna fórmula adicional, ni densificar nada. Esto se calcula igual de rápido usando solo las entradas no-cero de ambos vectores.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

fila_pos = train_x_vector[idx_pos]
fila_neg = train_x_vector[idx_neg]

# Producto punto directo sobre las matrices dispersas (aprovecha que ambas estan normalizadas a norma L2)
sim_manual = fila_pos.dot(fila_neg.T)[0, 0]
sim_sklearn = cosine_similarity(fila_pos, fila_neg)[0, 0]

print(f"Similitud (producto punto manual sobre vectores dispersos): {sim_manual:.4f}")
print(f"Similitud coseno (sklearn, para verificar):                 {sim_sklearn:.4f}")
print("\nUna similitud cercana a 0 indica que casi no comparten vocabulario relevante;")
print("cercana a 1 indicaría reseñas que usan prácticamente las mismas palabras clave.")

Similitud (producto punto manual sobre vectores dispersos): 0.0567
Similitud coseno (sklearn, para verificar):                 0.0567

Una similitud cercana a 0 indica que casi no comparten vocabulario relevante;
cercana a 1 indicaría reseñas que usan prácticamente las mismas palabras clave.


## 5. Entrenamiento del modelo — solo Naive Bayes

Como se indicó, en este notebook usamos **únicamente Multinomial Naive Bayes**:
$$ P(y \mid x) \propto P(y) \prod_{i=1}^{n} P(x_i \mid y)^{x_i} $$

En forma logarítmica (para evitar multiplicar muchísimas probabilidades pequeñas, lo cual causaría *underflow* numérico):
$$ \log P(y \mid x) \;\propto\; \log P(y) + \sum_{i=1}^{n} x_i \cdot \log P(x_i \mid y) $$

**Por qué esta fórmula es perfecta para una matriz dispersa:** si $x_i = 0$ (el término no aparece en la reseña), su contribución a la suma es cero — así que el modelo **solo necesita recorrer los índices no-cero** del vector, exactamente los que acabamos de leer en la sección 3. Nunca necesita "ver" las decenas de miles de columnas en cero.

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

modelo_nb = MultinomialNB()
modelo_nb.fit(train_x_vector, train_y)

pred = modelo_nb.predict(test_x_vector)
acc = accuracy_score(test_y, pred)

print(f"Exactitud en test: {acc*100:.2f}%")
print(classification_report(test_y, pred))

Exactitud en test: 84.55%
              precision    recall  f1-score   support

    negative       0.82      0.88      0.85      1650
    positive       0.87      0.81      0.84      1650

    accuracy                           0.85      3300
   macro avg       0.85      0.85      0.85      3300
weighted avg       0.85      0.85      0.85      3300



## 6. Verificación manual: cómo Naive Bayes "lee" el vector disperso de una reseña

Tomamos una reseña del conjunto de prueba y calculamos su predicción **a mano**, recorriendo solo los índices no-cero de su vector disperso (`fila.indices`, `fila.data`) y los parámetros ya aprendidos por el modelo (`feature_log_prob_`, `class_log_prior_`). Luego comparamos contra el resultado oficial de scikit-learn.

In [8]:
idx_test = 0
fila_test = test_x_vector[idx_test]
texto_test = test_x.iloc[idx_test]
real = test_y.iloc[idx_test]

print(f"Reseña de prueba (sentimiento real = {real}):")
print(f"  {texto_test[:200]}...\n")

clases = modelo_nb.classes_
log_prior = modelo_nb.class_log_prior_          # log P(y), una por clase
log_prob_features = modelo_nb.feature_log_prob_  # log P(termino | y), forma (n_clases, n_vocabulario)

indices_no_cero = fila_test.indices
valores_no_cero = fila_test.data

# --- Calculo manual: solo se recorren los terminos presentes en la reseña ---
log_verosimilitud_manual = np.array([
    log_prior[c] + np.sum(valores_no_cero * log_prob_features[c, indices_no_cero])
    for c in range(len(clases))
])

print("Log-verosimilitud conjunta (manual, antes de normalizar), por clase:")
for c, val in zip(clases, log_verosimilitud_manual):
    print(f"  {c}: {val:.4f}")

prediccion_manual = clases[np.argmax(log_verosimilitud_manual)]
print(f"\nPredicción manual:  {prediccion_manual}")
print(f"Predicción sklearn: {modelo_nb.predict(fila_test)[0]}")

# Normalizamos a probabilidades con el truco log-sum-exp (evita overflow/underflow numérico)
m = log_verosimilitud_manual.max()
proba_manual = np.exp(log_verosimilitud_manual - m)
proba_manual = proba_manual / proba_manual.sum()

proba_sklearn = modelo_nb.predict_proba(fila_test)[0]

print(f"\nProbabilidades (cálculo manual): {dict(zip(clases, np.round(proba_manual, 4)))}")
print(f"Probabilidades (sklearn):        {dict(zip(clases, np.round(proba_sklearn, 4)))}")
print(f"\n✔ Coinciden: {np.allclose(proba_manual, proba_sklearn)}")

Reseña de prueba (sentimiento real = positive):
  Now I myself had previously seen a few episodes of the Leauge Of Gentleman which I found hilarious. When I brought the film I was not sure if I knew enough about the series to get it, boy was I wrong....

Log-verosimilitud conjunta (manual, antes de normalizar), por clase:
  negative: -50.2122
  positive: -49.3917

Predicción manual:  positive
Predicción sklearn: positive

Probabilidades (cálculo manual): {np.str_('negative'): np.float64(0.3056), np.str_('positive'): np.float64(0.6944)}
Probabilidades (sklearn):        {np.str_('negative'): np.float64(0.3056), np.str_('positive'): np.float64(0.6944)}

✔ Coinciden: True


## 7. Conclusión

- Una matriz dispersa CSR se lee a través de 3 arreglos (`data`, `indices`, `indptr`); reconstruir una fila a mano coincide exactamente con la forma habitual de indexarla.
- El ahorro de memoria frente a una matriz densa es enorme (>99% menos espacio en este dataset), lo cual justifica su uso.
- Leímos 5 reseñas distintas directamente desde sus vectores dispersos: el número de términos no-cero refleja el **vocabulario único** de la reseña (tras quitar *stopwords*), no su longitud en palabras.
- La similitud coseno entre dos reseñas es simplemente el **producto punto** de sus vectores dispersos, gracias a la normalización L2 — otra operación que nunca necesita densificar nada.
- **Naive Bayes Multinomial** es un ejemplo perfecto de por qué el formato disperso es tan útil: su clasificación es una suma ponderada que **solo recorre los índices no-cero** del vector — lo verificamos replicando manualmente su cálculo interno y confirmando que coincide con `predict_proba`.

En este notebook se usó exclusivamente Naive Bayes, según lo solicitado. El notebook 1 ya lo compara contra Regresión Logística y SVM Lineal, si se quiere retomar esa comparación más adelante.